How to Calculate Support and Resistance Levels Using Python: A Step-by-Step Guide

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import MetaTrader5 as mt5


In [2]:
from datetime import datetime, timezone

if not mt5.initialize():
    raise RuntimeError(mt5.last_error())

symbols = mt5.symbols_get()
print(f"Total symbols: {len(symbols)}")
for s in symbols[:50]:  # first 50
    print(s.name)

# filter JPY/metals
watch = [s.name for s in symbols if any(k in s.name for k in ["JPY", "XAU", "XAG"])]
print("\nFiltered:", watch[:100])

# --- MT5 vs system time (MT5 = last tick time on a liquid symbol; aligns with broker feed clock)
sys_utc = datetime.now(timezone.utc)
sys_local = datetime.now().astimezone()

tick = None
tick_symbol = None
for s in symbols[:50]:
    if mt5.symbol_select(s.name, True):
        tick = mt5.symbol_info_tick(s.name)
        if tick is not None:
            tick_symbol = s.name
            break

print("\n--- Time comparison ---")
print(f"System now (UTC):   {sys_utc.isoformat()}")
print(f"System now (local): {sys_local.isoformat()}")
if tick is None:
    print("MT5 tick time:      (no tick — could not read)")
else:
    mt5_utc = datetime.fromtimestamp(int(tick.time), tz=timezone.utc)
    skew_s = (mt5_utc - sys_utc).total_seconds()
    print(f"MT5 tick time (UTC) via {tick_symbol}: {mt5_utc.isoformat()}")
    print(f"Skew (MT5 tick UTC − system UTC): {skew_s:+.1f} s  (large gaps often mean quiet market or clock drift)")

_ = mt5.shutdown()

Total symbols: 1101
AUDUSD
EURUSD
GBPUSD
USDJPY
EURCZK.i
EURHUF.i
EURTRY.i
EURZAR.i
GBPTRY.i
GBPZAR.i
USDCZK.i
USDHUF.i
USDMXN.i
USDTRY.i
USDZAR.i
AUDUSD.i
EURUSD.i
GBPUSD.i
NZDUSD.i
USDCAD.i
USDCHF.i
USDJPY.i
AUDCAD.i
AUDCHF.i
AUDJPY.i
AUDNZD.i
CADCHF.i
CADJPY.i
CHFJPY.i
EURAUD.i
EURCAD.i
EURCHF.i
EURGBP.i
EURJPY.i
EURNOK.i
EURNZD.i
EURSEK.i
GBPAUD.i
GBPCAD.i
GBPCHF.i
GBPJPY.i
GBPNZD.i
NZDCAD.i
NZDCHF.i
NZDJPY.i
USDDKK.i
USDNOK.i
USDSEK.i
ADIDAS.i
AIR-FRANCE.i

Filtered: ['USDJPY', 'USDJPY.i', 'AUDJPY.i', 'CADJPY.i', 'CHFJPY.i', 'EURJPY.i', 'GBPJPY.i', 'NZDJPY.i', 'XAGEUR.i', 'XAGUSD.i', 'XAUEUR.i', 'XAUUSD.i']

--- Time comparison ---
System now (UTC):   2026-05-27T11:19:01.595596+00:00
System now (local): 2026-05-27T14:49:01.595663+03:30
MT5 tick time (UTC) via AUDUSD: 2026-05-27T14:19:01+00:00
Skew (MT5 tick UTC − system UTC): +10799.4 s  (large gaps often mean quiet market or clock drift)


In [3]:
from pathlib import Path
from datetime import datetime, timezone

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SYMBOLS = [
    "GBPUSD","XAUUSD","GBPCAD","USDMXN","EURJPY","EURCAD","AUDCAD"
]

# IMPORTANT: M5 must come FIRST so it can be reused to top-up stale H1/D1.
# Some brokers (Errante demo seen here) publish H1/D1 with a multi-hour lag;
# the loop below detects that and appends a forming bar resampled from M5.
TIMEFRAMES = ['M5', 'M1', 'H1', 'D1']

DATE_FROM = None
DATE_TO = None
BARS = 1000000
BARS_BY_TF = {
    'M1':  BARS,
    'M5':  BARS,
    'M15': BARS,
    'M30': BARS,
    'H1':  100000,
    'H4':  30000,
    'D1':  5000,
}

MT5_TF = {
    'M1':  mt5.TIMEFRAME_M1,
    'M5':  mt5.TIMEFRAME_M5,
    'M15': mt5.TIMEFRAME_M15,
    'M30': mt5.TIMEFRAME_M30,
    'H1':  mt5.TIMEFRAME_H1,
    'H4':  mt5.TIMEFRAME_H4,
    'D1':  mt5.TIMEFRAME_D1,
}

# Resample rule / bar duration for the M5 top-up pass.
TF_RESAMPLE_RULE = {'H1': '1h', 'H4': '4h', 'D1': '1D'}
TF_BAR_DURATION  = {
    'H1': pd.Timedelta(hours=1),
    'H4': pd.Timedelta(hours=4),
    'D1': pd.Timedelta(days=1),
}


# ─────────────────────────────────────────────────────────────────────────────
# FETCH HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def fetch_mt5(symbol: str, tf: str, bars: int, date_from=None, date_to=None) -> pd.DataFrame:
    timeframe = MT5_TF[tf]
    if date_from is not None and date_to is not None:
        rates = mt5.copy_rates_range(symbol, timeframe, date_from, date_to)
    else:
        rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, bars)

    if rates is None or len(rates) == 0:
        raise RuntimeError(f'No data from MT5 for {symbol} {tf}. last_error={mt5.last_error()}')

    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s', utc=True)
    df['symbol'] = symbol
    df['timeframe'] = tf
    return df


def fetch_mt5_with_retry(symbol: str, tf: str, bars: int, date_from=None, date_to=None):
    trial_bars = bars
    last_exc = None
    for _ in range(4):
        try:
            return fetch_mt5(symbol, tf, bars=trial_bars, date_from=date_from, date_to=date_to), trial_bars
        except Exception as exc:
            last_exc = exc
            trial_bars = max(300, trial_bars // 2)
    raise RuntimeError(str(last_exc))


def resolve_symbol(base_symbol: str) -> str | None:
    for candidate in (base_symbol, f'{base_symbol}.i'):
        if mt5.symbol_select(candidate, True):
            return candidate
    return None


# ─────────────────────────────────────────────────────────────────────────────
# M5 TOP-UP — append a forming-bar resampled from M5 when broker H1/D1 is stale
# ─────────────────────────────────────────────────────────────────────────────
def topup_from_m5(higher_tf_df: pd.DataFrame, tf: str, m5_df: pd.DataFrame | None) -> pd.DataFrame:
    """
    Errante demo lags H1/D1 publication by hours-to-days. The live runner ALSO
    sees this lag in real time so it doesn't care; but BACKTESTS that compare
    against fresher M5 develop a divergence (NB31 found 520 such cases).

    Fix: if the broker's last H1/D1 bar is older than what M5 implies should
    exist by now, resample M5 onto the HTF grid and append the missing
    buckets. The native broker history is preserved verbatim; only forward
    extension is synthetic.

    No-op if M5 cache is missing or the TF is already current.
    """
    if m5_df is None or tf not in TF_RESAMPLE_RULE:
        return higher_tf_df

    rule          = TF_RESAMPLE_RULE[tf]
    bar_duration  = TF_BAR_DURATION[tf]
    last_htf_time = pd.Timestamp(higher_tf_df['time'].iloc[-1])
    last_m5_time  = pd.Timestamp(m5_df['time'].iloc[-1])

    # M5 doesn't even reach into the next HTF bucket → nothing to top up.
    if last_m5_time < last_htf_time + bar_duration:
        return higher_tf_df

    agg = {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'}
    for vol_col in ('tick_volume', 'real_volume'):
        if vol_col in m5_df.columns:
            agg[vol_col] = 'sum'
    if 'spread' in m5_df.columns:
        agg['spread'] = 'mean'

    resampled = (
        m5_df.set_index('time')
             .resample(rule, label='left', closed='left')
             .agg(agg)
             .dropna(subset=['open', 'high', 'low', 'close'])
             .reset_index()
    )

    new_bars = resampled[resampled['time'] > last_htf_time].copy()
    if new_bars.empty:
        return higher_tf_df

    new_bars['symbol']    = higher_tf_df['symbol'].iloc[0]
    new_bars['timeframe'] = tf

    # Align column order with the broker frame so concat is clean.
    for col in higher_tf_df.columns:
        if col not in new_bars.columns:
            new_bars[col] = 0
    new_bars = new_bars[higher_tf_df.columns]

    print(f'    + topped up {tf} with {len(new_bars)} bar(s) resampled from M5 '
          f'(native last: {last_htf_time}, m5 last: {last_m5_time})')
    return pd.concat([higher_tf_df, new_bars], ignore_index=True)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
if not mt5.initialize():
    raise RuntimeError(f'MT5 initialize() failed: {mt5.last_error()}')

base_dir = Path('./data')
results = {}
resolved_symbols = {}

try:
    for symbol in SYMBOLS:
        resolved = resolve_symbol(symbol)
        if resolved is None:
            print(f'Skip {symbol}: not available (tried {symbol} and {symbol}.i)')
            continue

        resolved_symbols[symbol] = resolved
        print(f'Using {symbol} -> {resolved}')

        # Per-symbol cache of the freshest M5 frame; reused to top-up H1/D1.
        m5_cache: pd.DataFrame | None = None

        for tf in TIMEFRAMES:
            tf_bars = BARS_BY_TF.get(tf, BARS)
            try:
                raw, used_bars = fetch_mt5_with_retry(
                    resolved, tf, bars=tf_bars,
                    date_from=DATE_FROM, date_to=DATE_TO,
                )

                if tf == 'M5':
                    m5_cache = raw.copy()

                raw = topup_from_m5(raw, tf, m5_cache)

                out_dir  = base_dir / symbol / tf
                out_dir.mkdir(parents=True, exist_ok=True)
                out_file = out_dir / 'ohlcv.csv'
                raw.to_csv(out_file, index=False)

                results[(symbol, tf)] = len(raw)
                print(f'Cached {symbol} ({resolved}) {tf}: {len(raw)} rows '
                      f'(bars={used_bars}, last={raw["time"].iloc[-1]}) -> {out_file}')
            except Exception as e:
                print(f'Failed {symbol} ({resolved}) {tf} (bars={tf_bars}): {e}')

    if results:
        preview_symbol, preview_tf = next(iter(results.keys()))
        preview_resolved = resolved_symbols[preview_symbol]
        preview_bars = BARS_BY_TF.get(preview_tf, BARS)
        preview, _ = fetch_mt5_with_retry(preview_resolved, preview_tf, bars=preview_bars,
                                          date_from=DATE_FROM, date_to=DATE_TO)
        preview.tail()
finally:
    mt5.shutdown()

Using GBPUSD -> GBPUSD
Cached GBPUSD (GBPUSD) M5: 1000000 rows (bars=1000000, last=2026-05-27 14:15:00+00:00) -> data\GBPUSD\M5\ohlcv.csv
Cached GBPUSD (GBPUSD) M1: 1000000 rows (bars=1000000, last=2026-05-27 14:19:00+00:00) -> data\GBPUSD\M1\ohlcv.csv
Cached GBPUSD (GBPUSD) H1: 100000 rows (bars=100000, last=2026-05-27 14:00:00+00:00) -> data\GBPUSD\H1\ohlcv.csv
Cached GBPUSD (GBPUSD) D1: 5000 rows (bars=5000, last=2026-05-27 00:00:00+00:00) -> data\GBPUSD\D1\ohlcv.csv
Using XAUUSD -> XAUUSD.i
Cached XAUUSD (XAUUSD.i) M5: 653489 rows (bars=1000000, last=2026-05-27 14:15:00+00:00) -> data\XAUUSD\M5\ohlcv.csv
Cached XAUUSD (XAUUSD.i) M1: 1000000 rows (bars=1000000, last=2026-05-27 14:19:00+00:00) -> data\XAUUSD\M1\ohlcv.csv
Cached XAUUSD (XAUUSD.i) H1: 61734 rows (bars=100000, last=2026-05-27 14:00:00+00:00) -> data\XAUUSD\H1\ohlcv.csv
Cached XAUUSD (XAUUSD.i) D1: 5000 rows (bars=5000, last=2026-05-27 00:00:00+00:00) -> data\XAUUSD\D1\ohlcv.csv
Using GBPCAD -> GBPCAD.i
Cached GBPCAD (GB